# デモ4：写真1枚から奥行きを推定する（Depth Anything V2）

写真1枚から、各画素がどれくらい近いかを推定します。人間は片目でも、物の大きさや重なり、影から奥行きを感じ取れます。このモデルも、大量の画像からそうした手がかりを学習しています。

- モデル：[depth-anything/Depth-Anything-V2-Small-hf](https://huggingface.co/depth-anything/Depth-Anything-V2-Small-hf)（Apache-2.0、約2500万パラメータ）
- 中の Transformer：画像を 14×14 のパッチに分けて読む ViT（DINOv2）。途中の層のトークンを画像の形に並べ直し、画素ごとの奥行きに広げる
- 演習2（`02_run_github_code.ipynb`）では、同じモデルを GitHub のコードから動かします

**最初に**：「ファイル」→「ドライブにコピーを保存」で自分のコピーを作り、「ランタイム」→「ランタイムのタイプを変更」で **GPU**（T4 など）を選んでから、上のセルから順に実行します。コードを読む必要はありません。何ができて、何が苦手かを見つけるのが目的です。

## 1. 準備

目安：約2分（うち計算 約1分）

In [ ]:
import importlib.util
import subprocess
import sys

# Colab に入っていない、または古いライブラリを入れる
if importlib.util.find_spec("transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate"], check=True)

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from skimage import data

device = "cuda" if torch.cuda.is_available() else "cpu"
# T4 は bfloat16 に対応していないので、GPU では float16 を使う
dtype = torch.float16 if device == "cuda" else torch.float32
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "（GPU を選ぶと速くなります）")

In [ ]:
from transformers import pipeline

depth_estimator = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf",
                           device=device, dtype=dtype)

## 2. 例の画像で試す

目安：約3分

In [ ]:
USE_UPLOAD = False  # True にすると、自分の画像をアップロードできる（JPEG か PNG）

# 例の画像（scikit-image に入っている写真。CC0 またはパブリックドメイン）
images = {
    "astronaut": Image.fromarray(data.astronaut()),
    "cat": Image.fromarray(data.chelsea()),
    "coffee": Image.fromarray(data.coffee()),
}
if USE_UPLOAD:
    import io
    from google.colab import files
    for name, content in files.upload().items():
        images[name] = Image.open(io.BytesIO(content)).convert("RGB")

fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4), squeeze=False)
for ax, (name, image) in zip(axes[0], images.items()):
    ax.imshow(image)
    ax.set_title(name)
    ax.axis("off")
plt.show()

右の図は推定した奥行きで、明るいほど近いことを表します（値は相対的なもので、単位はありません）。

In [ ]:
depths = {}
for name, image in images.items():
    depths[name] = np.array(depth_estimator(image)["predicted_depth"].float().cpu())
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(image)
    axes[0].set_title(name)
    axes[1].imshow(depths[name], cmap="inferno")
    axes[1].set_title("depth (bright = near)")
    for ax in axes:
        ax.axis("off")
    plt.show()

### 応用：近いところだけ残す

奥行きの値でしきい値を決めると、手前の物だけを切り抜けます。`NEAR_RATIO` を変えてみましょう。宇宙飛行士の写真で `NEAR_RATIO` を 0.4 まで下げると、顔が消えて胴体とヘルメットが残ります。顔は、手前に抱えたヘルメットや胸よりも少し奥にあるからです。

In [ ]:
NEAR_RATIO = 0.6  # ← 変えてみる（近いほうから何割の画素を残すか）

for name, image in images.items():
    depth = np.array(Image.fromarray(depths[name]).resize(image.size))  # 画像と同じ大きさにそろえる
    mask = depth >= np.quantile(depth, 1 - NEAR_RATIO)
    cutout = np.array(image) * mask[..., None]
    plt.figure(figsize=(5, 5))
    plt.imshow(cutout)
    plt.title(f"{name}: near {NEAR_RATIO:.0%}")
    plt.axis("off")
    plt.show()

## 3. やってみよう

目安：約10分

- 遠くまで続く風景、近くに物がある室内、鏡やガラス、絵や写真を撮った写真をアップロードして、得意と不得意を探します。
- 写真を撮った写真（壁に貼った写真など）は、本当は平らです。モデルはどう推定するでしょうか。